# 09 · Discrepancy and verification report

Reconcile README claims, telemetry, summary/seed JSON and camera-ready benchmarking framework, then compare implementation scope with the manuscript's dual-stream/LIG account. Presence in prose is not equivalent to evidence in logs.

In [ ]:
from pathlib import Path
import json, math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project():
    candidates=[Path.cwd(), Path.cwd()/"project_inputs", Path.cwd().parent/"project_inputs", Path("/mnt/data/diina_reproducibility/project_inputs"), Path("/mnt/data")]
    for p in candidates:
        if (p/"raw_results.csv").exists(): return p
    raise FileNotFoundError("Place notebook with project_inputs/ or set cwd to extracted project directory")
PROJECT=find_project()
df=pd.read_csv(PROJECT/"raw_results.csv")
summary=json.loads((PROJECT/"summary.json").read_text())
print("Project files:",PROJECT,"| telemetry shape:",df.shape,"| columns:",list(df.columns))
assert len(df)==1500 and set(df.columns)=={"seed","step","loss","fast_usage","slow_usage","entropy"}

In [ ]:
seed_json=json.loads((PROJECT/"result_seed_2022.json").read_text()); readme=(PROJECT/"README.md").read_text(); paper=(PROJECT/"NeurIPS_Camera_Ready_Final.txt").read_text(); script=(PROJECT/"DIINA_v19_1_LongDistance.py").read_text()
actual=sorted(df.seed.unique()); seed22=df[df.seed==2022].sort_values("step")
checks={"CSV seed set":actual,"rows":len(df),"README says 456":"456" in readme,"CSV has 456":456 in actual,"README says 1011":"1011" in readme,"CSV has 1011":1011 in actual,"CSV seed-2022 avg":seed22.loss.mean(),"JSON seed-2022 avg":seed_json["avg_loss"],"CSV seed-2022 final":seed22.loss.iloc[-1],"JSON seed-2022 final":seed_json["final_loss"],"summary avg_loss":summary["avg_loss"],"paper framework title present":"DIINA Benchmarking Framework" in paper,"v19.1 uses GRU":"nn.GRU" in script,"v19.1 gate":"self.inhibit" in script}
for k,v in checks.items(): print(f"{k}: {v}")

In [ ]:
print("\nMismatch checks:")
print("Seeds in README but not CSV (expected README five-seed list):",sorted({42,123,789,1011,2022}-set(actual)))
print("CSV values vs seed JSON:",{"avg_delta":float(seed22.loss.mean()-seed_json["avg_loss"]),"final_delta":float(seed22.loss.iloc[-1]-seed_json["final_loss"])})
print("Metric units note: summary.avg_loss mean matches pooled CSV average, while its reported std is not the across-five seed-mean SD.")
print("Camera-ready text frames a benchmark framework; manuscript.docx describes a dual-stream neural architecture; benchmark code is a GRU proxy.")

**Evidence status:** CSV and JSON are directly testable here; README and papers are claims/context. The user-facing code sources expose simplified synthetic tasks and baseline implementations; they do not establish the broad low-resource language, FLOPs, or manuscript architectural claims from the telemetry alone.